# 03 Dynamic Agents
Up until now, our Agents have been powerful, but they have been fairly _static_. Fixed tools, fixed prompts, fixed models. _Real-world AI applications_ don't work like that in production! 

For example, we may have to adjust _behavior_ of our Agent depending on whether the end-user is an employee or an external-user OR if her default language preference is different (say Spanish) from our default (say English). Building a one-size-fits-all model is just fine as an MVP (or in Development). However, when it comes time to deploy the Agent in front of real users (in Production), we'll need an Agent that can _dynamically_ adjust itself.

In this notebook we'll cover how you can adjust system prompts, tools, and even the underlying models of the Agent on-the-fly - even multiple-times during a conversation! To achieve all this, we'll build custom middleware (as you might have guessed already 😎).

In [16]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

In the first notebook [01_middleware.ipynb](01_middleware.ipynb) of this series, we saw how to use custom middleware to _trim_ our chat context, to avoid context overrun.

<div align="center"> 
<img src="images/node_style_middleware.png" width="250" heigh="250" alt="Node style middleware"/> 
</div>

Middleware such as the `SummarizationMiddleware`, which gets called before/after agent/model/tool calls is called **Node-style middleware**. Node-style middleware is really good at hooking into the various stages of Agent execution and changes its state - great for use-cases like modifying messages (user prompts) and entire conversations at runtime.

However, this is _not_ dynamic behavior. To achieve what we are envisaging, we need to tap even deeper and hook into the model or tool instance itself. This isn't _before_ or _after_ the model/tool(s) - this _is_ the model/tool(s) itself. To achieve this, we'll be wrapping our model/tool(s) itself with custom middleware functions, which are aptly named **Wrap-style middleware**. The middleware functions we define for our will be annotated with `@wrap_model_call`/`@awrap_model_call` or `@wrap_tool_call`/`@awrap_tool_call` annotations.

The model instance is represented as a _model request_. 

<div align="center"> 
<img src="images/model_request.png" width="250" heigh="250" alt="Model request"/> 
</div>

This contains information like the system prompt, available tool calls, the state and the foundation model itself. So if you _grab_ the model request, you are getting access to all the features of the model - tools, state & the foundation model itself. This will let you do things like swapping underlying OpenAI LLM with a Claude model, add/remove specialized tools based on _current_ user permission (for example, an employee using the chatbot gets access to more tools than an external user), even system prompts.

Let's see how that works in practice. 

### Modifying the System Prompt at runtime

As a first example, we'll see how we can _dynamically adjust the System prompt_. In this example, we'll assume an Engish speaking & a Spanish speaking user interacting with the same chatbot. While the former will see responses in English, the latter would like responses in Spanish. In a real-scenario, we'd hook into some API that stores/reads user preferences, such as _preferred language_ and then feed the same to our Agent. 

Refer to inline comments in the code, which will explain how to modify system prompts at runtime.

In [17]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest


# Step1: define a dataclass to hold all runtime context (parameters)
# in this example, it's the user's preferred language, which we set in
# our program. In production, imagine we call some API that retrieves the
# user's preferences and from there we get the user's preferred language
@dataclass
class ModelContext:
    user_language: str


# Step2: define the function to modify the system prompt at runtime.
# This function is annotated with a @dynamic_prompt annotation
# It takes an instance of langchain.agents.middleware.ModelRequest as it's only param
# It returns a string, which is the modified System prompt
@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """generates dynamic system prompt based on user's preferred language"""

    # the ModelRequest gives me access to the runtime context
    context: ModelContext = request.runtime.context
    # and now we can access all the fields of the context
    preferred_language: str = context.user_language
    default_system_prompt: str = "You are a helpful assistant. "

    if preferred_language.lower().strip() != "english":
        default_system_prompt = (
            default_system_prompt + f"ONLY respond in {preferred_language}."
        )

    print(
        f" ---- in user_language_prompt(...). System prompt -> {default_system_prompt} ---- "
    )
    return default_system_prompt

In [18]:
from langchain.agents import create_agent

# Step3: create our agent using the usual create_agent() call
#  - tell the agent the data type of our context via the context_schema param
#  - tell the model of the function that will modify the system prompt, via the
#       middleware parameter
agent = create_agent(
    model="openai:gpt-5-nano",
    # my context schema is an instance of ModelContext type dataclass
    context_schema=ModelContext,
    # and this is the function that will modify the system prompt
    middleware=[user_language_prompt],
)

In [23]:
from langchain.messages import HumanMessage

In [ ]:
# Step4: simulate chat with same Agent

# Step4a - simulate English user, whose preferred language is English
response = agent.invoke(
    {"messages": [HumanMessage("Hello, how are you?")]},
    # use default value that is set
    context={"user_language": "English"},
)
print(response["messages"][-1].content)

# note the intermediate output from our user_language_prompt function

 ---- in user_language_prompt(...). System prompt -> You are a helpful assistant.  ---- 
Hello! I’m here and ready to help. How can I assist you today?


In [25]:
# Step4b - simulate Spanish user, whose preferred language is Spanish
response = agent.invoke(
    # saying the same thing "Hello, How are you?" in Spanish
    {"messages": [HumanMessage("¡Hola, ¿cómo estás?")]},
    # context={"user_language": "Spanish"} is also ok!
    context=ModelContext(user_language="Spanish"),
)
print(response["messages"][-1].content)

 ---- in user_language_prompt(...). System prompt -> You are a helpful assistant. ONLY respond in Spanish. ---- 
¡Hola! Estoy bien, gracias. ¿Y tú? ¿En qué puedo ayudarte hoy?


In [27]:
# Step4c - simulate an Indian user chatting in Hindi
response = agent.invoke(
    {"messages": [HumanMessage("नमस्ते, आप कैसे हैं?")]},
    context=ModelContext(user_language="Hindi"),
)
print(response["messages"][-1].content)

 ---- in user_language_prompt(...). System prompt -> You are a helpful assistant. ONLY respond in Hindi. ---- 
नमस्ते! मैं ठीक हूँ, धन्यवाद। आप कैसे हैं? अगर किसी चीज़ में मदद चाहिए या सवाल हो, तो बताइए।


## Chatbot

In this section, let's build a chatbot that is accessed by both internal users (employees) and external users.

The purpose of this example is to show you how we can dynamically control the tools that are available to the model based on whether the user is an internal or external user. Internal users get access to web search as well as database search (pretent this is some internal database for the purpose of illustration). External users get access to ONLY web search - they can't access internal stuff! The tools are set dynamically depending on the context provided to agent, which determines if user is _internal_ or _external_.

In [28]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()
# assume this is the internal database, to which external users
# won't get access. In production this could be an employee or sales
# or some other internal database.
db = SQLDatabase.from_uri("sqlite:///db/chinook.db")
db_schema = db.get_table_info()


# define tools for our chatbot
@tool
def web_search(query: str) -> Dict[str, Any]:
    """search the web for information"""
    print(f" --- web_search({query}) tool called --- ")
    return tavily_client.search(query)


@tool
def sql_query(query: str) -> str:
    """obtain information from database using SQL queries"""
    print(f" --- sql_query({query}) tool called --- ")
    try:
        return db.run(query)
    except Exception as e:
        return f"Database error: {e}"

In [3]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from dataclasses import dataclass


# call that dynamically assigns the tools a model can use
# depending on whether the user is an internal or external user
@wrap_model_call
def dynamic_tool_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """dynamically call tools based on runtime context"""
    user_role = request.runtime.context.user_role
    print(f" --- in dynamic_tool_call -> user_role = {user_role} --- ")

    if user_role.lower().strip() != "internal":
        # external user -> will get access to ONLY web-search
        tools = [web_search]
        # modify the tools used by model
        request = request.override(tools=tools)

    return handler(request)


@dataclass
class UserRole:
    user_role: str = "internal"

In [30]:
from langchain.agents import create_agent

# create our agent
agent = create_agent(
    model="openai:gpt-5-nano",
    # by default, we assume user is an employee
    # who gets access to all tools
    tools=[web_search, sql_query],
    # this middleware will modify the tools list
    middleware=[dynamic_tool_call],
    # schema that decided if user is internal or not
    context_schema=UserRole,
)

In [37]:
from langchain_core.messages import HumanMessage

# web-search query, should work for both internal & external users
web_query = HumanMessage(
    "Who is the current PM of Canada, as of April 2026. Just return the name"
)
# ONLY internal users get access to database info
database_query = HumanMessage("How many artists in the database?")

In [38]:
# let's try web query with internal user. It should work!
response = agent.invoke(
    {"messages": [web_query]},
    context={"user_role": "internal"},
)
print(response["messages"][-1].content)

 --- in dynamic_tool_call -> user_role = internal --- 
 --- web_search(Canada Prime Minister April 2026) tool called --- 
 --- in dynamic_tool_call -> user_role = internal --- 
Mark Carney


In [39]:
# Next, let's try a database query with internal user - this should also work!
response = agent.invoke(
    {"messages": [database_query]},
    context={"user_role": "internal"},
)
print(response["messages"][-1].content)

 --- in dynamic_tool_call -> user_role = internal --- 
 --- sql_query(SELECT COUNT(*) AS artist_count FROM artists;) tool called --- 
 --- in dynamic_tool_call -> user_role = internal --- 
275 artists. If you’d like a breakdown (e.g., by genre, country, or active status), I can run more queries.


In [40]:
# let's try web query with EXTERNAL user. It should work!
response = agent.invoke(
    {"messages": [web_query]},
    context={"user_role": "internal"},
)
print(response["messages"][-1].content)

 --- in dynamic_tool_call -> user_role = internal --- 
 --- web_search(Canada current prime minister April 2026) tool called --- 
 --- in dynamic_tool_call -> user_role = internal --- 
Mark Carney


In [41]:
# Next, let's try a database query with internal user - this should FAIL!
response = agent.invoke(
    {"messages": [user_query]},
    context={"user_role": "external"},
)
print(response["messages"][-1].content)

 --- in dynamic_tool_call -> user_role = external --- 
I’m missing context about which database you mean. Could you tell me the database or dataset (e.g., PostgreSQL, MySQL, MongoDB, Excel/CSV, etc.) and what you want counted (all rows, or distinct artists by name/id, and whether to include inactive/archived records)?

Here are quick examples you can use:

- SQL (PostgreSQL/MySQL/SQLite)
  - Total rows: SELECT COUNT(*) AS artist_count FROM artists;
  - Distinct by name (to avoid duplicates): SELECT COUNT(DISTINCT name) AS artist_count FROM artists;
  - Distinct by id: SELECT COUNT(DISTINCT artist_id) AS artist_count FROM artists;

- MongoDB (NoSQL)
  - Total documents: db.artists.countDocuments({});
  - Distinct by artist_id: db.artists.distinct("artist_id").length;
  - Active-only: db.artists.countDocuments({ active: true });

- CSV/Excel (in Python)
  - Basic: import pandas as pd; df = pd.read_csv('artists.csv'); len(df)
  - Distinct by id: df['artist_id'].nunique()

If you share the

### Switching Models
In this example we show how we can dynamically change the underlying model (language model) used by our Agent.

Why would you want to change the underlying model at runtime??

In [42]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from dataclasses import dataclass
from langchain.chat_models import init_chat_model

In [53]:
# let's define 2 models

large_model = init_chat_model("anthropic:claude-sonnet-4-5")
standard_model = init_chat_model("openai:gpt-5-nano")


@wrap_model_call
def state_based_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """select model based on state conversation length"""
    message_count = len(request.messages)
    model = large_model if message_count > 10 else standard_model
    print(f" --- in state_based_mode -> message_count: {message_count} model: {model}")
    request = request.override(model=model)
    return handler(request)

In [54]:
# create our agent
from langchain.agents import create_agent

agent = create_agent(
    model=standard_model,  # by default
    middleware=[state_based_model],
    system_prompt="You are roleplaying like a real life helpful office assistant",
)

In [48]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("Did you water the office plant today?")]},
)
print(f"\n{'-'*80}")
print(response["messages"][-1].content)

 --- in state_based_mode -> message_count: 1 model: profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001ECAF940680> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001ECAF2D3170> root_client=<openai.OpenAI object at 0x000001ECAF39AC60> root_async_client=<openai.AsyncOpenAI object at 0x000001ECAF51F5F0> model_name='gpt-5-nano' mode

In [50]:
# here is another way to tell which model generated the above response
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [58]:
# now let's simulate a longer (>10 messages) conversation
from langchain.messages import AIMessage

messages = [
    HumanMessage(content="Good morning! Can you check what meetings I have today?"),
    AIMessage(
        content="Good morning! You have a team standup at 9 AM, a client call at 11 AM, and a project review at 3 PM."
    ),
    HumanMessage(content="Can you send a reminder to the team about the 9 AM standup?"),
    AIMessage(
        content="Done! A reminder has been sent to the team for the 9 AM standup."
    ),
    HumanMessage(
        content="Also, can you book the small meeting room for the project review at 3 PM?"
    ),
    AIMessage(
        content="The small meeting room is now booked for your 3 PM project review."
    ),
    HumanMessage(content="Great. Are there any pending approvals in my inbox?"),
    AIMessage(
        content="Yes, you have two pending approvals — a budget request from the design team and a leave application from one of your direct reports."
    ),
    HumanMessage(content="Please approve both and notify the respective teams."),
    AIMessage(
        content="Both approvals have been processed and the design team and your direct report have been notified."
    ),
    HumanMessage(content="Can you also compile a list of all deadlines for this week?"),
    AIMessage(
        content="Here's the list: Monday — project status update, Wednesday — vendor invoice submission, Friday — monthly performance report and client proposal draft."
    ),
    HumanMessage(
        content="Can you draft a quick status update email for the project and send it to the stakeholders?"
    ),
    AIMessage(
        content="Draft ready and sent! The stakeholders have been updated on the current project status, key milestones, and upcoming deliverables."
    ),
    HumanMessage(
        content="What is the best way to handle the vendor invoice submission on Wednesday?"
    ),
]

print(f"We have {len(messages)} messages in the conversation")

We have 15 messages in the conversation


In [59]:
# NOTE: the last HumanMessage in the above conversation
# response from the following invoke() call should answer that question

response = agent.invoke(
    {"messages": messages},
)
print(f"\n{'-'*80}")
print(response["messages"][-1].content)

 --- in state_based_mode -> message_count: 15 model: profile={'name': 'Claude Sonnet 4.5 (latest)', 'release_date': '2025-09-29', 'last_updated': '2025-09-29', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'structured_output': True} model='claude-sonnet-4-5' max_tokens=64000 anthropic_api_url='https://api.anthropic.com' anthropic_api_key=SecretStr('**********') model_kwargs={}

--------------------------------------------------------------------------------
Here are my recommendations for the vendor invoice submission:

1. **Review now** - Check all vendor invoices today to identify any m

In [60]:
# here is another way to tell which model generated the above response
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-5-20250929
